<a href="https://colab.research.google.com/github/ProfAI/machine-learning-fondamenti/blob/main/4%20-%20La%20Classificazione/multiclass_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# La Classificazione Multiclasse

La **Classificazione Multiclasse** si applica a problemi in cui la variabile target presenta $K > 2$ classi discrete (ad esempio, classificare le immagini di numeri da $0$ a $9$).

Per estendere algoritmi intrinsecamente binari (come la regressione logistica classica) a compiti multiclasse, si usano principalmente due strategie:
- **One-vs-Rest (OvR / One-vs-All)**: Addestra $K$ classificatori binari indipendenti. Il classificatore $c$-esimo viene addestrato a distinguere la classe $c$ da tutte le altre $K-1$ classi raggruppate insieme.
- **One-vs-One (OvO)**: Addestra $K(K-1)/2$ classificatori binari per ogni coppia possibile di classi. Durante l'inferenza, la classe finale viene determinata tramite uno schema di voto a maggioranza.


In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

### Generiamo il dataset

In [3]:
X, y = make_classification(n_samples=100, n_features=4, n_informative=4, n_redundant=0, n_repeated=0, n_classes=3, random_state=0)

### Creiamo il modello e valuatiamo il modello - Tecnica OvR

In Scikit-learn, la regressione logistica applica l'approccio One-vs-Rest impostando il parametro `multi_class='ovr'`.

### Performance dell'approccio OvR:
- **Accuratezza**: Otteniamo una $\text{Train Accuracy} \approx 0.67$ e una $\text{Test Accuracy} \approx 0.77$.
- **ROC AUC (OVO)**: Otteniamo un $\text{Train AUC} \approx 0.853$ e un $\text{Test AUC} \approx 0.841$.

--- 

## 📊 Guida all'interpretazione del Classification Report

Il `classification_report` di Scikit-learn mostra un riepilogo dettagliato delle metriche di classificazione per ciascuna classe:

### 1. Le Colonne Principali
- **Precision (Precisione)**: Per una determinata classe, risponde alla domanda: *"Di tutti gli elementi che il modello ha classificato come appartenenti a questa classe, quanti ne appartengono effettivamente?"*
  $$\text{Precision}_c = \frac{\text{TP}_c}{\text{TP}_c + \text{FP}_c}$$
  *Esempio (Test Set)*: Per la classe `0`, la precision è $0.80$ (l'80% delle predizioni per la classe 0 era corretto).
- **Recall (Sensibilità / Richiamo)**: Risponde alla domanda: *"Di tutti gli elementi reali appartenenti a questa classe, quanti ne ha individuati il modello?"*
  $$\text{Recall}_c = \frac{\text{TP}_c}{\text{TP}_c + \text{FN}_c}$$
  *Esempio (Test Set)*: Per la classe `0`, la recall è $0.92$ (il modello ha individuato il 92% degli elementi reali appartenenti alla classe 0).
- **F1-Score**: È la media armonica di Precision e Recall per quella classe. Fornisce una singola misura di sintesi bilanciata, utile soprattutto in caso di classi fortemente sbilanciate.
  $$F_{1,c} = 2 \times \frac{\text{Precision}_c \times \text{Recall}_c}{\text{Precision}_c + \text{Recall}_c}$$
- **Support (Supporto)**: Indica il numero di campioni reali (osservazioni reali) appartenenti a ciascuna classe in quella specifica partizione (es. $22$ campioni per la classe 0 nel Train set, $13$ nel Test set).

### 2. Le Righe di Sintesi (Medie Generali)
- **Accuracy (Accuratezza)**: Rappresenta la frazione totale di predizioni corrette fatte dal modello su tutte le classi:
  $$\text{Accuracy} = \frac{\text{Predizioni Corrette}}{\text{Totale Campioni}}$$
  *Nota*: Nel Test Set otteniamo un'accuratezza del $77\%$ ($0.77$).
- **Macro Avg (Media Macro)**: Calcola la media aritmetica semplice (non pesata) delle metriche (Precision, Recall, F1) calcolate per ogni singola classe:
  $$\text{Macro Avg} = \frac{1}{K} \sum_{c=1}^K \text{Metric}_c$$
  *Nota*: Tratta tutte le classi allo stesso modo, indipendentemente dal loro supporto. Utile se si vuole dare uguale importanza alle classi anche se alcune sono molto rare.
- **Weighted Avg (Media Ponderata)**: Calcola la media delle metriche pesata in base al supporto (numero di campioni) di ciascuna classe:
  $$\text{Weighted Avg} = \sum_{c=1}^K \left( \frac{\text{Support}_c}{\text{Totale Campioni}} \times \text{Metric}_c \right)$$
  *Nota*: Questa media rispecchia maggiormente la distribuzione reale delle classi nel dataset.


In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.3)

Inizializziamo il modello di **Regressione Logistica**. Nonostante il nome, è un algoritmo di classificazione che modella la probabilità di appartenenza a una classe tramite la funzione Sigmoide:
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

In [5]:
lr = LogisticRegression(multi_class="ovr")
lr.fit(X_train, y_train)

LogisticRegression(multi_class='ovr')

Il `classification_report` ci fornisce in un unico specchietto le metriche fondamentali: Precision, Recall e F1-Score, calcolate per ciascuna classe.

In [6]:
y_pred_train = lr.predict(X_train)
y_pred_test = lr.predict(X_test)
print("TRAIN REPORT")
print(classification_report(y_train, y_pred_train))
print("TEST REPORT")
print(classification_report(y_test, y_pred_test))

TRAIN REPORT
              precision    recall  f1-score   support

           0       0.77      0.91      0.83        22
           1       0.62      0.57      0.59        23
           2       0.61      0.56      0.58        25

    accuracy                           0.67        70
   macro avg       0.67      0.68      0.67        70
weighted avg       0.66      0.67      0.66        70

TEST REPORT
              precision    recall  f1-score   support

           0       0.80      0.92      0.86        13
           1       0.71      0.56      0.63         9
           2       0.75      0.75      0.75         8

    accuracy                           0.77        30
   macro avg       0.75      0.74      0.74        30
weighted avg       0.76      0.77      0.76        30



Procediamo con la prossima fase di esplorazione ed esecuzione del codice:

In [15]:
y_proba_train = lr.predict_proba(X_train)
roc_auc_score(y_train, y_proba_train, multi_class="ovo", average="macro")

0.8529249011857708

Procediamo con la prossima fase di esplorazione ed esecuzione del codice:

In [16]:
y_proba_test = lr.predict_proba(X_test)
roc_auc_score(y_test, y_proba_test, multi_class="ovo", average="macro")

0.8409900284900286

### Creiamo il modello e valuatiamo il modello - Tecnica Multinomial

L'approccio **Multinomial** (noto anche come *Softmax Regression*) rappresenta la naturale generalizzazione della regressione logistica al caso multiclasse. Invece di addestrare modelli binari indipendenti, calcola contemporaneamente le probabilità per tutte le classi tramite la funzione **Softmax**:
$$P(y = c \mid X) = \frac{e^{z_c}}{\sum_{j=1}^K e^{z_j}}$$

In questo modo, la somma delle probabilità predette per le $K$ classi su ciascuna istanza è esattamente pari a $1$.

### Performance dell'approccio Multinomial:
- **Accuratezza**: Otteniamo una $\text{Train Accuracy} \approx 0.69$ e una $\text{Test Accuracy} \approx 0.70$.
- **ROC AUC (OVO)**: Otteniamo un $\text{Train AUC} \approx 0.851$ e un $\text{Test AUC} \approx 0.844$.


In [18]:
lr = LogisticRegression(multi_class="multinomial")
lr.fit(X_train, y_train)

LogisticRegression(multi_class='multinomial')

Il `classification_report` ci fornisce in un unico specchietto le metriche fondamentali: Precision, Recall e F1-Score, calcolate per ciascuna classe.

In [19]:
y_pred_train = lr.predict(X_train)
y_pred_test = lr.predict(X_test)
print("TRAIN REPORT")
print(classification_report(y_train, y_pred_train))
print("TEST REPORT")
print(classification_report(y_test, y_pred_test))

TRAIN REPORT
              precision    recall  f1-score   support

           0       0.80      0.91      0.85        22
           1       0.61      0.61      0.61        23
           2       0.64      0.56      0.60        25

    accuracy                           0.69        70
   macro avg       0.68      0.69      0.69        70
weighted avg       0.68      0.69      0.68        70

TEST REPORT
              precision    recall  f1-score   support

           0       0.73      0.85      0.79        13
           1       0.67      0.44      0.53         9
           2       0.67      0.75      0.71         8

    accuracy                           0.70        30
   macro avg       0.69      0.68      0.67        30
weighted avg       0.70      0.70      0.69        30



Procediamo con la prossima fase di esplorazione ed esecuzione del codice:

In [20]:
y_proba_train = lr.predict_proba(X_train)
roc_auc_score(y_train, y_proba_train, multi_class="ovo", average="macro")

0.8513043478260869

Procediamo con la prossima fase di esplorazione ed esecuzione del codice:

In [22]:
y_proba_test = lr.predict_proba(X_test)
roc_auc_score(y_test, y_proba_test, multi_class="ovo", average="macro")

0.8438390313390314